In [ ]:
!pip install --quiet --upgrade gspread gspread_dataframe mne scipy
from google.colab import auth
auth.authenticate_user()

import gspread, os, json, warnings, numpy as np, pandas as pd, scipy.signal as sig
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.auth import default
import mne

warnings.filterwarnings("ignore", category=RuntimeWarning)

creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
spreadsheet = gc.open("Copy of Artifact Annotation")
worksheet = spreadsheet.sheet1
artifact_df = get_as_dataframe(worksheet).dropna(how='all')
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData/'
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()])
print(subject_dirs)

In [ ]:
# Helper to record artifact annotation
def record_artifact(subj, sess_file, ra_name, times, freqs, artifact_type):
    # Ensure inputs are arrays
    times = np.atleast_1d(times)
    freqs = np.atleast_1d(freqs)

    # Format safely
    formatted_times = []
    for t in times:
        try:
            formatted_times.append(f"{float(t):.2f}s")
        except (ValueError, TypeError):
            formatted_times.append(str(t))

    formatted_freqs = [str(f) for f in freqs]

    return {
        "Subject": subj,
        "Session File": sess_file,
        "RA Name": ra_name,
        "Artifact Times": ', '.join(formatted_times) if len(formatted_times) > 0 else '',
        "Artifact Frequencies": ', '.join(formatted_freqs) if len(formatted_freqs) > 0 else '',
        "Artifact Type": artifact_type
    }
    return new_row

In [ ]:
# Process EEG files and apply filters
records = []

for subj in subject_dirs:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.set')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        print(f"Processing: {session_path}")
        try:
            raw = mne.io.read_raw_eeglab(session_path, preload=True)

            # 1. High-pass filter
            raw.filter(0.1, None)
            records.append(record_artifact(subj, session_file, "Auto", "Full recording", ["<0.1"], "High-pass filter (0.1 Hz)"))

            # 2. Notch filter
            raw.notch_filter(60)
            records.append(record_artifact(subj, session_file, "Auto", "Full recording", [60], "Notch filter (60 Hz)"))

            # 3. Detect EOG events (blinks)
            eog_events = mne.preprocessing.find_eog_events(raw, event_id=998, l_freq=1, h_freq=10, ch_name=None)
            blink_times = raw.times[eog_events[:, 0]]  # Convert sample indices to times
            records.append(record_artifact(subj, session_file, "Auto", blink_times, ["1–10"], "Blink (EOG)"))

        except Exception as e:
            print(f"Failed to process {session_path}: {e}")

# Update spreadsheet
new_df = pd.DataFrame(records)
updated_df = pd.concat([artifact_df, new_df], ignore_index=True)
worksheet.clear()
set_with_dataframe(worksheet, updated_df)
print("Annotation spreadsheet updated.")

In [ ]:
# Test with a single file
records = []

test_subj = '10'  # Assuming '10' is a valid subject directory
test_session_file = '10_001.set'

test_subj_path = os.path.join(root_dir, test_subj)
test_session_path = os.path.join(test_subj_path, test_session_file)

print(f"Processing single test file: {test_session_path}")

try:
    raw = mne.io.read_raw_eeglab(test_session_path, preload=True)

    # 1. High-pass filter
    raw.filter(0.1, None)
    records.append(record_artifact(test_subj, test_session_file, "Auto", "Full recording", ["<0.1"], "High-pass filter (0.1 Hz)"))

    # 2. Notch filter
    raw.notch_filter(60)
    records.append(record_artifact(test_subj, test_session_file, "Auto", "Full recording", [60], "Notch filter (60 Hz)"))

    # 3. Detect EOG events (blinks)
    eog_proxy_candidates = ['E1', 'E32', 'E17', 'E25']

    eog_channel = None
    for candidate in eog_proxy_candidates:
        if candidate in raw.ch_names:
            eog_channel = candidate
            break

    if eog_channel:
        eog_events = mne.preprocessing.find_eog_events(
            raw,
            event_id=998,
            l_freq=1,
            h_freq=10,
            ch_name=eog_channel
        )
        blink_times = raw.times[eog_events[:, 0]]
        records.append(record_artifact(
            subj, session_file, "Auto", blink_times, ["1–10"],
            f"Blink (EOG via {eog_channel})"
    ))
    else:
        print(f"⚠️ No EOG proxy found in {session_path}")

except FileNotFoundError:
        print(f"Error: Test file not found at {test_session_path}")
except Exception as e:
        print(f"Failed to process {test_session_path}: {e}")

# Optionally, print the records collected for this file
print("\nRecords collected for the test file:")
for record in records:
    display(record)